# Airbnb Madrid: Que factores influyen en el precio y que diferencia a los Superhosts?

**Autora:** Juliana Castano Yarce  
**Fecha:** Mayo 2026  
**Curso:** Programacion y analisis reproducible con datos — Proyecto Final

## 1. Introduccion y Pregunta de Analisis

### Dataset
- **Fuente:** [Inside Airbnb](https://insideairbnb.com/) — un proyecto independiente que recopila datos publicos de Airbnb
- **Ciudad:** Madrid, Espana
- **Fecha de descarga:** 6 de mayo de 2026
- **Tamano:** ~25,000 alojamientos (listings) con 79 variables

### Pregunta de analisis

> **Que factores (barrio, tipo de habitacion, reviews, caracteristicas del host) influyen en el precio de un Airbnb en Madrid, y que diferencia a los Superhosts del resto?**

### Por que es interesante?

Madrid es una de las ciudades mas turisticas de Europa, y Airbnb ha transformado su mercado de alojamiento. Entender que determina el precio y que hace a un buen host puede servir para:

- **Viajeros:** saber si estan pagando un precio justo
- **Hosts:** entender como posicionar mejor su alojamiento
- **La ciudad:** analizar como se distribuye el turismo por barrios

---
## 2. Carga y Exploracion Inicial

In [ ]:
# Importar librerias
import pandas as pd       # para trabajar con tablas de datos
import numpy as np        # para operaciones matematicas
import matplotlib.pyplot as plt  # para hacer graficos
import seaborn as sns     # para graficos mas bonitos
import warnings
warnings.filterwarnings('ignore')  # ocultar avisos que no nos afectan

In [ ]:
# Cargar el dataset
df = pd.read_csv('../data/listings.csv')

# Ver cuantas filas y columnas tiene
print(f"El dataset tiene {df.shape[0]:,} filas y {df.shape[1]} columnas")

In [ ]:
# Ver las primeras 5 filas para hacernos una idea de los datos
df.head()

In [ ]:
# Ver los nombres de las columnas y sus tipos de datos
# object = texto, float64/int64 = numeros
df.info()

In [ ]:
# Estadisticas basicas de las columnas numericas (promedios, minimos, maximos, etc.)
df.describe().round(2)

### Primeras observaciones

- El dataset tiene **25,000 alojamientos** con **79 columnas** — hay mucha informacion disponible
- La mayoria son **apartamentos completos** (~67%), seguidos de **habitaciones privadas** (~32%)
- La columna `price` tiene **~6,000 valores faltantes** — tendremos que limpiar eso en la siguiente seccion
- Algunas columnas como `bedrooms` y `review_scores_rating` tambien tienen valores faltantes
- Hay columnas que estan casi vacias (como `calendar_updated` y `license`) — las podemos ignorar

---
## 3. Limpieza y Transformacion

In [ ]:
# Paso 1: Seleccionar solo las columnas que necesitamos
columnas = ['name', 'neighbourhood_cleansed', 'room_type', 'accommodates',
            'bedrooms', 'beds', 'price', 'minimum_nights',
            'number_of_reviews', 'review_scores_rating',
            'host_is_superhost', 'host_response_rate', 'host_total_listings_count']

df = df[columnas]
print(f"Ahora tenemos {df.shape[1]} columnas en vez de 79")
df.head()

In [ ]:
# Paso 2: Convertir el precio de texto ("$157.00") a numero (157.0)
# Quitamos el signo $ y las comas, y lo convertimos a numero
df['price'] = df['price'].str.replace('$', '', regex=False)
df['price'] = df['price'].str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'])

print("Ejemplo de precios despues de limpiar:")
print(df['price'].head(10))

In [ ]:
# Paso 3: Ver cuantos valores nulos hay por columna
print("Valores nulos por columna:")
print("-" * 40)
print(df.isnull().sum())

In [ ]:
# Paso 4: Eliminar filas que no tienen precio (sin precio no podemos analizar nada)
antes = len(df)
df = df.dropna(subset=['price'])
print(f"Filas antes: {antes:,}")
print(f"Filas despues: {len(df):,}")
print(f"Eliminadas: {antes - len(df):,} (no tenian precio)")

In [ ]:
# Paso 5: Convertir host_response_rate de texto ("95%") a numero (95)
df['host_response_rate'] = df['host_response_rate'].str.replace('%', '', regex=False)
df['host_response_rate'] = pd.to_numeric(df['host_response_rate'])

# Convertir host_is_superhost de "t"/"f" a True/False
# "t" = true (si es superhost), "f" = false (no es superhost)
df['host_is_superhost'] = df['host_is_superhost'].map({'t': True, 'f': False})

print("host_response_rate ejemplo:", df['host_response_rate'].head(3).tolist())
print("host_is_superhost ejemplo:", df['host_is_superhost'].head(3).tolist())

In [ ]:
# Paso 6: Eliminar precios extremos que no tienen sentido
# Un Airbnb a $0 es un error, y uno a $10,000/noche no es tipico
antes = len(df)
df = df[(df['price'] > 10) & (df['price'] < 1000)]
print(f"Filas antes: {antes:,}")
print(f"Filas despues: {len(df):,}")
print(f"Eliminadas: {antes - len(df):,} (precios menores a $10 o mayores a $1,000)")
print(f"\nRango de precios final: ${df['price'].min():.0f} - ${df['price'].max():.0f}")

### Resumen de limpieza

| Paso | Que hicimos | Por que |
|------|-------------|---------|
| 1 | Seleccionar 13 columnas de 79 | Solo necesitamos las relevantes para nuestra pregunta |
| 2 | Convertir precio de texto a numero | `"$157.00"` no se puede usar en calculos |
| 3 | Eliminar filas sin precio | Sin precio no podemos analizar nada |
| 4 | Convertir response rate y superhost | Necesitamos numeros y valores claros, no texto |
| 5 | Eliminar precios extremos (<$10, >$1,000) | Son errores o casos atipicos que distorsionan el analisis |